# Dynamics of Adversarial Attractors

In this notebook, we explore the mapping dynamics and confidence shifts caused by adversarial attractors. 

Rather than just counting the top sink classes, we will visualize two key aspects:
1. **Transition Mapping:** By plotting the Original Class Index vs. the Adversarial Class Index, we can visually identify attractors as dense horizontal bands (multiple diverse source classes converging into a single target class ID).
2. **Confidence Shift:** We analyze if the neural network is more or less confident in its wrong adversarial predictions compared to its original correct predictions.

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import seaborn as sns

# Import models
from tensorflow.keras.applications import EfficientNetB0, InceptionV3, MobileNetV2

# Import preprocessing and decoding
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff, decode_predictions as decode_eff
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inc, decode_predictions as decode_inc
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob, decode_predictions as decode_mob

In [ ]:
# Dictionary with model configurations
models_dict = {
    'MobileNetV2': {
        'model': MobileNetV2(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_mob,
        'decode_fn': decode_mob,
        'eps_scale': 1.0
    },
    'EfficientNetB0': {
        'model': EfficientNetB0(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_eff,
        'decode_fn': decode_eff,
        'eps_scale': 127.5
    },
    'InceptionV3': {
        'model': InceptionV3(weights='imagenet'),
        'target_size': (299, 299),
        'preprocess_fn': preprocess_inc,
        'decode_fn': decode_inc,
        'eps_scale': 1.0
    }
}

# Freeze weights as we only need to perform inference and compute gradients w.r.t inputs
for config in models_dict.values():
    config['model'].trainable = False

### Helper Functions & Data Loading

In [ ]:
def load_and_preprocess_image(img_path, target_size, preprocess_fn):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = preprocess_fn(img) # Dynamic preprocessing
    img = tf.expand_dims(img, axis=0) # Add batch dimension
    return img

In [ ]:
# Define the path to the 100 images folder
dataset_path = '../miniimagenet_random_100/'
image_paths = glob.glob(os.path.join(dataset_path, '*.*'))

# Limit to 100 just in case
image_paths = image_paths[:100]
print(f"Total images loaded for the experiment: {len(image_paths)}")

In [ ]:
loss_object = tf.keras.losses.CategoricalCrossentropy()

def fgsm_attack(input_image, input_label, epsilon, current_model):
    with tf.GradientTape() as tape:
        tape.watch(input_image)
        prediction = current_model(input_image)
        loss = loss_object(input_label, prediction)
    
    # Get gradients of the loss w.r.t to the input image
    gradient = tape.gradient(loss, input_image)
    
    # Get the sign of the gradients
    signed_grad = tf.sign(gradient)
    
    # Create the adversarial image
    adv_image = input_image + epsilon * signed_grad
    return adv_image

### Execute Attack and Record Dynamics
For each model, we will record the original class ID, the adversarial class ID, and the confidence scores for both.

In [ ]:
base_epsilon = 0.01

# Dictionary to store results for plotting
dynamics_data = {}

for model_name, m_config in models_dict.items():
    print(f"Processing {model_name}...")
    current_model = m_config['model']
    epsilon = base_epsilon * m_config['eps_scale']
    
    orig_indices = []
    adv_indices = []
    orig_confidences = []
    adv_confidences = []
    
    for img_path in image_paths:
        input_img = load_and_preprocess_image(img_path, m_config['target_size'], m_config['preprocess_fn'])
        
        # Original Prediction
        orig_probs = current_model(input_img, training=False)
        orig_idx = tf.argmax(orig_probs, axis=-1).numpy()[0]
        orig_conf = orig_probs[0, orig_idx].numpy()
        
        # Attack
        input_label = tf.reshape(tf.one_hot(orig_idx, orig_probs.shape[-1]), (1, -1))
        adv_img = fgsm_attack(input_img, input_label, epsilon, current_model)
        
        # Adversarial Prediction
        adv_probs = current_model(adv_img, training=False)
        adv_idx = tf.argmax(adv_probs, axis=-1).numpy()[0]
        adv_conf = adv_probs[0, adv_idx].numpy()
        
        # Store data (only if the attack was successful to see true transitions)
        if orig_idx != adv_idx:
            orig_indices.append(orig_idx)
            adv_indices.append(adv_idx)
            orig_confidences.append(orig_conf)
            adv_confidences.append(adv_conf)
            
    dynamics_data[model_name] = {
        'orig_idx': orig_indices,
        'adv_idx': adv_indices,
        'orig_conf': orig_confidences,
        'adv_conf': adv_confidences
    }

print("Data collection complete!")

### Visualization: Transition Mapping & Confidence Shift

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(f'Attractor Dynamics (Untargeted FGSM, $\varepsilon$ = {base_epsilon})', fontsize=18, fontweight='bold', y=1.02)

for col, (model_name, data) in enumerate(dynamics_data.items()):
    
    # Row 1: Transition Mapping (Scatter Plot)
    ax_scatter = axes[0, col]
    ax_scatter.scatter(data['orig_idx'], data['adv_idx'], alpha=0.6, c='purple', edgecolors='k', s=50)
    
    # Draw a diagonal line y=x. Points would fall here if attack failed. Since we filter successful attacks, points will be off-diagonal
    ax_scatter.plot([0, 1000], [0, 1000], 'k--', alpha=0.3, label='No Change (y=x)')
    
    ax_scatter.set_title(f'{model_name}: Class Transitions', fontsize=14, fontweight='bold')
    ax_scatter.set_xlabel('Original Class Index (0-999)')
    ax_scatter.set_ylabel('Adversarial Class Index (0-999)')
    ax_scatter.set_xlim(0, 1000)
    ax_scatter.set_ylim(0, 1000)
    
    # Highlight horizontal bands (attractors)
    ax_scatter.text(0.5, -0.15, "Horizontal patterns indicate sink classes", 
                    transform=ax_scatter.transAxes, ha='center', fontsize=10, fontstyle='italic')

    # Row 2: Confidence Shift (Boxplot)
    ax_box = axes[1, col]
    
    # Format data for seaborn
    plot_data = []
    for oc, ac in zip(data['orig_conf'], data['adv_conf']):
        plot_data.append({'Type': 'Original Image', 'Confidence': oc})
        plot_data.append({'Type': 'Adversarial Image', 'Confidence': ac})
        
    df = pd.DataFrame(plot_data)
    
    if not df.empty:
        sns.boxplot(x='Type', y='Confidence', data=df, ax=ax_box, palette=['#66b3ff', '#ff9999'], width=0.5)
        sns.stripplot(x='Type', y='Confidence', data=df, ax=ax_box, color='black', alpha=0.4, jitter=True)
        
    ax_box.set_title(f'{model_name}: Confidence Shift', fontsize=14, fontweight='bold')
    ax_box.set_ylabel('Prediction Confidence (0.0 - 1.0)')
    ax_box.set_xlabel('')
    ax_box.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.show()